In [1]:
# Task 1 - Vectorize It
import numpy as np
import time
prices = [10, 25, 30, 45, 50, 55, 60, 70, 80, 90,
          15, 20, 35, 40, 65, 75, 85, 95, 100, 110]
loop_prices = [p * 1.08 for p in prices]
print("Loop result (first 5):", loop_prices[:5])
arr = np.array(prices)
vectorized_prices = arr * 1.08
print("Vectorized result (first 5):", vectorized_prices[:5])
print("Results match:", np.allclose(loop_prices, vectorized_prices))
threshold = 50
above_threshold = vectorized_prices[vectorized_prices > threshold]
print(f"\nTaxed prices above {threshold}:", above_threshold)
grid = np.array([
    [10, 20, 30, 40],
    [50, 60, 70, 80],
    [90, 100, 110, 120]
])
print("\n2D array:\n", grid)
print("Row sums   (axis=1):", grid.sum(axis=1))
print("Column sums(axis=0):", grid.sum(axis=0))
big_list = list(range(1, 1_000_001))
big_arr  = np.array(big_list)
start = time.perf_counter()
_ = [x * 1.08 for x in big_list]
loop_time = time.perf_counter() - start
start = time.perf_counter()
_ = big_arr * 1.08
vec_time = time.perf_counter() - start
print(f"\nLoop time:       {loop_time:.4f}s")
print(f"Vectorized time: {vec_time:.4f}s")
print(f"Speedup:         {loop_time / vec_time:.1f}x faster")

Loop result (first 5): [10.8, 27.0, 32.400000000000006, 48.6, 54.0]
Vectorized result (first 5): [10.8 27.  32.4 48.6 54. ]
Results match: True

Taxed prices above 50: [ 54.   59.4  64.8  75.6  86.4  97.2  70.2  81.   91.8 102.6 108.  118.8]

2D array:
 [[ 10  20  30  40]
 [ 50  60  70  80]
 [ 90 100 110 120]]
Row sums   (axis=1): [100 260 420]
Column sums(axis=0): [150 180 210 240]

Loop time:       0.2358s
Vectorized time: 0.0384s
Speedup:         6.1x faster


In [2]:
# Task 2 - Load & Explore
import pandas as pd
import io
csv_data = """name,department,salary,years_experience
Alice,Engineering,95000,5
Bob,Engineering,85000,3
Carol,Engineering,110000,8
Dave,Marketing,60000,2
Eve,Marketing,70000,4
Frank,Marketing,65000,3
Grace,HR,55000,6
Henry,HR,50000,1
Ivy,HR,58000,4
Jack,Engineering,120000,10
Kate,Marketing,75000,6
Leo,HR,52000,2
Mia,Engineering,98000,6
Nina,Marketing,68000,3
Oscar,Engineering,105000,7
"""
with open("employees.csv", "w") as f:
    f.write(csv_data)
df = pd.read_csv("employees.csv")
print("=== df.info() ===")
df.info()
print("\n=== df.describe() ===")
print(df.describe())
bool_filter = df[df["department"] == "Engineering"]
print("\n--- Engineering (boolean indexing) ---")
print(bool_filter[["name", "salary"]])
query_filter = df.query("department == 'Engineering'")
print("\n--- Engineering (.query()) ---")
print(query_filter[["name", "salary"]])
print("Both filters match:", bool_filter.equals(query_filter))
summary = df.groupby("department")["salary"].agg(
    avg_salary="mean",
    headcount="count"
)
summary = summary.sort_values("avg_salary", ascending=False)
print("\n=== Department Summary (sorted by avg salary) ===")
print(summary.round(0))

=== df.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              15 non-null     object
 1   department        15 non-null     object
 2   salary            15 non-null     int64 
 3   years_experience  15 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 612.0+ bytes

=== df.describe() ===
              salary  years_experience
count      15.000000         15.000000
mean    77733.333333          4.666667
std     22848.778231          2.497618
min     50000.000000          1.000000
25%     59000.000000          3.000000
50%     70000.000000          4.000000
75%     96500.000000          6.000000
max    120000.000000         10.000000

--- Engineering (boolean indexing) ---
     name  salary
0   Alice   95000
1     Bob   85000
2   Carol  110000
9    Jack  120000
12    Mia   98000
14  Oscar  105000

--- Engi

In [3]:
# Task 3 - Join the Data
import pandas as pd
import numpy as np
employees = pd.read_csv("employees.csv")
departments = pd.DataFrame({
    "department":        ["Engineering", "Marketing", "HR"],
    "department_budget": [500000, 200000, 150000]
})
inner = pd.merge(employees, departments, on="department", how="inner")
print("=== Inner Join ===")
print(inner[["name", "department", "department_budget"]].head())
new_employee = pd.DataFrame([{
    "name": "Zara", "department": "Legal", "salary": 90000, "years_experience": 4
}])
employees_extended = pd.concat([employees, new_employee], ignore_index=True)
left = pd.merge(employees_extended, departments, on="department", how="left")
print("\n=== Left Join (Zara has NaN budget — Legal not in departments) ===")
print(left[left["name"] == "Zara"])   # NaN in department_budget
employees.loc[2, "salary"] = np.nan
employees.loc[7, "years_experience"] = np.nan
print("\n=== Missing value counts ===")
print(employees.isna().sum())
mean_salary = employees["salary"].mean()
employees["salary"] = employees["salary"].fillna(mean_salary)
mean_exp = employees["years_experience"].mean()
employees["years_experience"] = employees["years_experience"].fillna(mean_exp)
print("\n=== After filling NaNs ===")
print(employees.isna().sum())
print(employees[["name", "salary", "years_experience"]].to_string())

=== Inner Join ===
    name   department  department_budget
0  Alice  Engineering             500000
1    Bob  Engineering             500000
2  Carol  Engineering             500000
3   Dave    Marketing             200000
4    Eve    Marketing             200000

=== Left Join (Zara has NaN budget — Legal not in departments) ===
    name department  salary  years_experience  department_budget
15  Zara      Legal   90000                 4                NaN

=== Missing value counts ===
name                0
department          0
salary              1
years_experience    1
dtype: int64

=== After filling NaNs ===
name                0
department          0
salary              0
years_experience    0
dtype: int64
     name         salary  years_experience
0   Alice   95000.000000          5.000000
1     Bob   85000.000000          3.000000
2   Carol   75428.571429          8.000000
3    Dave   60000.000000          2.000000
4     Eve   70000.000000          4.000000
5   Frank   65000.0

In [4]:
# Task 4 - Tell a Story
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
df = pd.read_csv("employees.csv")
avg_salary = df.groupby("department")["salary"].mean().sort_values(ascending=False)
plt.figure(figsize=(7, 4))
avg_salary.plot(kind="bar", color=["steelblue", "salmon", "seagreen"])
plt.title("Average Salary by Department")
plt.xlabel("Department")
plt.ylabel("Average Salary ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("chart_1_bar.png")
plt.close()
print("Saved: chart_1_bar.png")
colors = {"Engineering": "steelblue", "Marketing": "salmon", "HR": "seagreen"}
plt.figure(figsize=(7, 4))
for dept, group in df.groupby("department"):
    plt.scatter(group["years_experience"], group["salary"],
                label=dept, color=colors[dept], s=80)
plt.title("Experience vs Salary by Department")
plt.xlabel("Years of Experience")
plt.ylabel("Salary ($)")
plt.legend()
plt.tight_layout()
plt.savefig("chart_2_scatter.png")
plt.close()
print("Saved: chart_2_scatter.png")
plt.figure(figsize=(7, 4))
plt.hist(df["salary"], bins=6, color="mediumpurple", edgecolor="white")
plt.title("Salary Distribution")
plt.xlabel("Salary ($)")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.savefig("chart_3_histogram.png")
plt.close()
print("Saved: chart_3_histogram.png")
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
headcount = [10, 12, 13, 15, 18, 20]
plt.figure(figsize=(7, 4))
plt.plot(months, headcount, marker="o", color="darkorange", linewidth=2)
plt.title("Monthly Headcount Growth")
plt.xlabel("Month")
plt.ylabel("Headcount")
plt.tight_layout()
plt.savefig("chart_4_line.png")
plt.close()
print("Saved: chart_4_line.png")

Saved: chart_1_bar.png
Saved: chart_2_scatter.png
Saved: chart_3_histogram.png
Saved: chart_4_line.png


In [5]:
# Task 5 - Consolidation Reflection
# 1. What Java habits did you have to unlearn or adjust?
"""
I had to stop over-structuring everything like Java. Python OOP feels simpler, FastAPI handles a lot through Pydantic, and with Pandas/NumPy I had to get used to vectorized operations instead of loops.
"""
# 2. Where will you use one skill again?
"""
I think vectorized NumPy operations will come up again in ML and DL. We'll probably use them a lot when working with datasets and doing calculations on large amounts of data.
"""
# 3. What topic are you least confident about?
"""
I'm least confident with testing FastAPI endpoints right now. Before the assessment, I'll build a small API and actually write tests for it instead of just reading about it.
"""

"\nI'm least confident with testing FastAPI endpoints right now. Before the assessment, I'll build a small API and actually write tests for it instead of just reading about it.\n"

In [6]:
# Task 6 -
# 1. Class with constructor and a dunder method (from memory)
class Student:
    def __init__(self, name):
        self.name = name
    def __str__(self):
        return self.name
# 2. FastAPI endpoint with a Pydantic request model (from memory)
from fastapi import FastAPI
from pydantic import BaseModel
app = FastAPI()
class User(BaseModel):
    name: str
    age: int
@app.post("/user")
def create_user(user: User):
    return {"message": f"Created user {user.name}"}
# 3. Pandas groupby().agg(...) call (from memory)
df.groupby("department").agg(
    average_salary=("salary", "mean"),
    total_employees=("salary", "count")
)

# After checking the materials
"""
I didn't find any major mistakes, but I had to double-check the exact agg() syntax.

That was mainly a syntax issue because I remembered the concept but wasn't fully sure about the named aggregation format.
"""

ModuleNotFoundError: No module named 'fastapi'